In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

## Make dataset (json)

In [ ]:
DATA_DIR = Path("./../data")
# list all images in the development set
image_paths = list((DATA_DIR / "development_set").glob("*.jpg"))
image_names = [img_path.name for img_path in image_paths]

all_images_paths = list((DATA_DIR / "v3_our_table" / "train").glob("*.jpg"))
all_image_names = [img_path.name for img_path in all_images_paths]

missing_images = [img_name for img_name in all_image_names if img_name not in image_names]
missing_images = [str(Path("v3_our_table") / "train" / img_name) for img_name in missing_images]

json_output = json.dumps({"image_path": missing_images}, indent=4)

## Test solution

In [ ]:
OUTPUT_JSON = DATA_DIR / "json" / "output.json"
ANNOTATIONS = DATA_DIR / "v3_our_table" / "train" / "_annotations.createml.json"

with open(str(OUTPUT_JSON), "r") as f:
    out_data = json.load(f)

with open(str(ANNOTATIONS), "r") as f:
    annotations_data = json.load(f)

def is_striped(number):
    return number > 8 and number <= 15

def _to_px_coords(ball, *, w: int, h: int):
    x1 = int(round(ball["xmin"] * w))
    y1 = int(round(ball["ymin"] * h))
    x2 = int(round(ball["xmax"] * w))
    y2 = int(round(ball["ymax"] * h))

    # clamp to image bounds
    x1 = max(0, min(w - 1, x1))
    x2 = max(0, min(w - 1, x2))
    y1 = max(0, min(h - 1, y1))
    y2 = max(0, min(h - 1, y2))

    # ensure top-left / bottom-right ordering
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1

    return (x1, y1), (x2, y2)

def img_data_2_img(img_data, data_dir=DATA_DIR):
    image = cv2.imread(str(data_dir / img_data['image_path']))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w, _ = image.shape

    for ball in img_data["balls"]:
        pt1, pt2 = _to_px_coords(ball, w=w, h=h)
        cv2.rectangle(image, pt1, pt2, (255, 255, 255), 3)
        cv2.putText(
            image, 
            f"{ball['number']}, {"st" if is_striped(ball['number']) else "sl"}", 
            (pt1[0], pt1[1] - 10), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2
        )

    cv2.putText(image, f"Num balls: {img_data['num_balls']}", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255, 0, 0), 3)

    return image


n = len(out_data)

for img in out_data[:3]:
    image = img_data_2_img(img)
    plt.figure(figsize=(12, 6))
    plt.imshow(image)
    ax = plt.gca()
    ax.set_title(f"Image: {img['image_path']}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
ALL_DATA = DATA_DIR / "v3_original/train/"
results = []

for img in annotations_data:
    d = {"image_path": img["image"], "num_balls": 0, "balls": []}

    cv2_img = cv2.imread(str(ALL_DATA / img["image"]))
    h, w, _ = cv2_img.shape

    for annotation in img["annotations"]:
        if annotation["label"] == "Dot":
            continue

        d["num_balls"] += 1
        d["balls"].append({
            "number": 0 if annotation["label"] == "Cue" else 1 if annotation["label"] == "Solid" else 9,
            "xmin": (annotation["coordinates"]["x"]) / w - 0.01,
            "xmax": (annotation["coordinates"]["x"] + annotation["coordinates"]["width"]) / w - 0.01,
            "ymin": (annotation["coordinates"]["y"]) / h - 0.01,
            "ymax": (annotation["coordinates"]["y"] + annotation["coordinates"]["height"]) / h - 0.01
        })

    results.append(d)

with open(str("../data/json/v3_expectes_output.json"), "w") as f:
    json.dump(results, f, indent=4)

In [ ]:
for image_name in image_names:
    img_data = [img for img in results if img["image_path"] == image_name][0]
    image_with_boxes = img_data_2_img(img_data, data_dir=DATA_DIR / "development_set")
    plt.figure(figsize=(20, 20))
    plt.imshow(image_with_boxes)
    plt.title(f"Image: {img_data['image_path'].split('.')[0]}")
    plt.axis('off')
    plt.show()
    break

In [ ]:
for output in out_data[:3]:
    print(f"Image: {output['image_path']},\n Num balls: {output['num_balls']}")
    for ball in output["balls"]:
        print(f"  Ball: {ball['number']}, ({ball['xmin']:.2f}, {ball['ymin']:.2f}) to ({ball['xmax']:.2f}, {ball['ymax']:.2f})")

    expected_output = [res for res in results if res["image_path"] in output["image_path"]][0]
    print(f"Expected Num balls: {expected_output['num_balls']}")
    for ball in expected_output["balls"]:
        print(f"  Expected: {ball['number']}, ({ball['xmin']:.2f}, {ball['ymin']:.2f}) to ({ball['xmax']:.2f}, {ball['ymax']:.2f})")
    print("\n" + "-"*50 + "\n")